# 07 · Follow-ups: example-level forgetting arm and head-bias check
### Machine Unlearning is Extinction, Not Forgetting

The main (class-level) experiments found that unlearned models keep the forgotten class hidden but do **not**
relapse under the classic triggers. A 2025 study ("From Dormant to Deleted", Siddiqui et al.) found the opposite
for forgetting **individual examples**: plain fine-tuning on the retained data brought the forgotten examples
back. This notebook runs our relapse tests on example-level forgetting, with the same original models, methods
and code, so the paper can show the boundary directly instead of citing it. It also runs a quick check of how
much of the class-level forgetting sits in the output layer's bias.

| Stage | What | Jobs | Time on T4 x2 |
|---|---|---|---|
| **H** head-bias check | For every class-level model of notebook 03: is the forgotten class suppressed by its output bias? Put the original bias back; shift that output by one constant | ~360 | ~10-15 min |
| **A** retrain | Train from scratch without each forget set: 3 seeds × 3 random forget sets + 1 pilot set | 10 | ~45 min |
| **B** calibrate | Hyper-parameter grid per method on the pilot forget set; ranks the settings | ~33 | ~40 min |
| **C** unlearn | 5 methods × 9 forget sets (+ 5 pilot), each tuned until **matched** | 50 | ~1 h |
| **D** relapse tests | Spontaneous recovery (retain fine-tuning, noise, quantization, pruning, BN recompute), renewal (4 contexts), savings (relearn part of the forget set, score the rest) | 63 | ~2 h |

### Example-level forgetting in one paragraph
A random **10 %** of the training set (5,000 images, about 50 per class) is to be forgotten; every class stays.
The retrained model never saw those images, so it recognises them only as well as unseen test images. The
original model recognises them almost perfectly, because it trained on them. Our score is therefore
**`mem_gap` = accuracy on the forgotten images − accuracy on test images**: about 0 for the retrained model,
about 0.25 for the original. *Relapse* = `mem_gap` rising again after a manipulation, beyond the retrained model.

### The matching rule here
An unlearned model counts if its accuracy on the forgotten images is **within 3 points** of the retrained
model's (above or below: over-forgetting is as detectable as under-forgetting) and its test accuracy is within
3 points. Calibration uses a separate pilot forget set (draw 9), never the nine analysed ones.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Needed by | Why |
|---|---|---|
| `ext-code` (**version 1.3**) | all stages | The shared library; 1.3 adds the example-level arm and the head-bias check. Class-level code is unchanged (verified bit-identical to 1.2). |
| CIFAR-100 (`fedesoriano/cifar100`, or `ext-02-retrain`, which carries `data_cache/`) | all stages | Check that the setup cell prints `CIFAR-100: found at …`. |
| `ext-01-originals` | all stages | The three original models are the starting point of every unlearning run here, and the reference for the head-bias check. |
| `ext-02-retrain`, `ext-03-unlearning` | stage H only | The class-level models whose output layer the head-bias check inspects. |
| *(when resuming)* this notebook's previous output (`ext-07-example-level`) | – | Skips finished jobs and carries their files over. |

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

The shared settings (seeds, classes, model, training recipe, `norm`) must equal notebook 01's; the setup checks
this automatically. Only the new example-level settings are explained here.

| Parameter | Value | What it does and why this value |
|---|---|---|
| `xl_frac` | 0.1 | Share of the training set to forget (5,000 images). 10 % random forgetting is the standard example-level benchmark (SalUn, SCRUB papers). |
| `xl_draws` | `[0, 1, 2]` | Three independent random forget sets per original model: 3 seeds × 3 draws = **9 units per method**, enough for a paired Wilcoxon test (smallest possible p = 0.002). |
| `xl_pilot_draw` | 9 | A separate forget set (seed 0) used only for calibration, then excluded, as the pilot class `lamp` was at class level. |
| `xl_match_gap` | 0.03 | Matching rule: accuracy on the forgotten images within ±3 points of the retrained model's. |
| `xl_sv_shots` | `[100, 500]` | Savings: relearn from this many forgotten images (plus replay) and score the *remaining* forgotten images. If relearning 2-10 % of the forget set brings back the advantage on the rest, the knowledge was dormant. |
| `sr_*`, `ctx_synthetic`, `sv_*`, `recovery_bn_modes`, `bn_recal_sizes`, `collapse_frac` | as notebook 04 | Identical relapse procedures, so class and example level are directly comparable. CIFAR-100-C is not used here: it has corrupted versions of the *test* images only, not of the forgotten training images. |
| `RUN_H` … `RUN_D` | `True` | Switch stages on or off. Stage H is independent of A-D. |

In [ ]:
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    methods=["FT", "NegGrad+", "RL", "SCRUB", "SalUn"],
    forget_batch_size=64, clip_grad=5.0, match_retain_gap=0.03, max_tries=3,
    aba_classes=["maple_tree", "tiger", "rose", "shark", "bus"], aba_context="gray",
    mctx_classes=["maple_tree", "tiger", "rose", "shark", "bus"], mctx_contexts=["clean", "gray", "blur"],
    # relapse tests (identical to notebook 04)
    recovery_bn_modes=["update", "frozen"], bn_recal_sizes=[10, 100, 1000, 10000],
    sr_epochs=5, sr_lr=0.01, sr_noise=[0.005, 0.01, 0.02, 0.05], sr_noise_reps=3,
    sr_quant_bits=[8, 6, 4], sr_prune=[0.2, 0.5, 0.7], ctx_synthetic=["gray", "blur", "noise", "contrast"],
    sv_steps=100, sv_lr=0.01, sv_replay_bs=128, sv_eval_steps=[0, 1, 2, 5, 10, 20, 50, 100], collapse_frac=0.5,
    # example-level arm
    xl_frac=0.1, xl_draws=[0, 1, 2], xl_pilot_draw=9, xl_match_gap=0.03, xl_sv_shots=[100, 500],
    # infrastructure
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
RUN_H, RUN_A, RUN_B, RUN_C, RUN_D = True, True, True, True, True
U.check_experiment_config(CFG)
assert U.VERSION >= "1.3", "Attach ext-code version 1.3 (this notebook needs the example-level arm)."

## Stage H: head-bias check (class level)

A May 2026 paper (Zheng et al., "Classification-Head Bias in Class-Level Machine Unlearning") showed that many
class-unlearning methods mainly push the forgotten class's **output bias** down. This stage measures that for our
300 class-level models, against the retrained and original references:

- `bias_z`: the forgotten class's bias compared with the other 99 (standard scores; very negative = suppressed by bias),
- `bias_restore_forget_acc`: forget-class accuracy after putting back **only the original bias** (one number),
- `row_restore_forget_acc`: the same after putting back its whole output row,
- `shift_recall`: forget-class recall after adding the one constant to that output that lets 1 % of retain
  training images flip to it. If this is high, a single number undoes the unlearning.

Needs `ext-02-retrain` and `ext-03-unlearning` attached.

In [ ]:
if RUN_H:
    jobs = U.jobs_headbias(CFG)
    print(f"{len(jobs)} models for the head-bias check")
    U.launch("headbias", jobs, CFG)

## Stage A: retrained references (example level)

For each seed and forget set, a model is trained from scratch on the other 90 % of the data, with the same
recipe and seed as the original. Its accuracy on the 5,000 forgotten images is the target every unlearned model
must match. The forget sets are drawn deterministically from (seed, draw), so every stage and notebook sees the
same images.

In [ ]:
if RUN_A:
    U.launch("xl_retrain", U.jobs_xl_retrain(CFG), CFG)
    xr = U.load_df("xl_retrain", CFG)
    if len(xr):
        display(xr[["seed", "draw", "forget_acc", "test_acc", "mem_gap", "mia_auc"]].round(4))

## Stage B: calibration on the pilot forget set

Every grid setting is applied to the original model (seed 0) for the pilot forget set and compared with its
retrained model. Settings are ranked per method: matched first (highest test accuracy), then by how close the
accuracy on the forgotten images is to the retrained model's. The top `max_tries` go to stage C.

In [ ]:
if RUN_B:
    jobs = U.jobs_xl_calibrate(CFG)
    print(f"{len(jobs)} calibration jobs")
    U.launch("xl_calibrate", jobs, CFG)
    xl_candidates, xl_table = U.xl_select_candidates(CFG)
    display(xl_table.round(4))
xl_candidates = U.load_xl_candidates(CFG)
for m, c in xl_candidates.items():
    print(f"{m:9s} tries in order: {c}")

## Stage C: example-level unlearning

For each method, seed and forget set: unlearn with the first candidate, check the matching rule, try the next
candidate if needed (up to `max_tries`), keep the best. The pilot forget set is unlearned too (5 models) but is
excluded from every analysis.

In [ ]:
if RUN_C:
    U.launch("xl_unlearn", U.jobs_xl_unlearn(CFG, xl_candidates, pilot=True) + U.jobs_xl_unlearn(CFG, xl_candidates), CFG)

## Stage D: relapse tests (example level)

The same procedures as notebook 04, scored with `mem_gap` (and the other example-level measures), for every
unlearned model of the nine analysed forget sets, its retrained model and the original model:

- **spontaneous recovery**: 5 epochs of retain fine-tuning (each checkpoint scored with updated and with original
  BatchNorm statistics), weight noise, quantization, pruning, and BatchNorm recompute alone;
- **renewal**: the forgotten and the test images shown in gray, blur, noise and low contrast;
- **savings**: relearn from 100 or 500 of the forgotten images and score the other forgotten images.

The prediction from the literature is that retain fine-tuning brings the forgotten examples back here, unlike
at class level.

In [ ]:
if RUN_D:
    jobs = U.jobs_xl_extinction(CFG)
    print(f"{len(jobs)} models to test")
    U.launch("xl_extinction", jobs, CFG)

## Quick look

Matching summary, and the headline number for each stage. The statistics are in notebook 08.

In [ ]:
import pandas as pd
xu = U.load_df("xl_unlearn", CFG)
if len(xu):
    main = xu[xu["draw"] != CFG["xl_pilot_draw"]]
    print("example-level unlearning - matched rate and accuracies (means over 9 forget sets):")
    display(main.groupby("method")[["matched", "forget_acc", "ref_forget_acc", "test_acc", "ref_test_acc", "mem_gap", "leak"]].mean().round(4))
xe = U.load_df("xl_extinction", CFG)
if len(xe):
    ids = ["kind", "method", "seed", "draw"]
    base = xe[xe["test"] == "base"][ids + ["mem_gap"]].rename(columns={"mem_gap": "before"})
    rf = xe[(xe["test"] == "spontaneous") & (xe["proxy"] == "retain_ft") & (xe["level"] == xe["level"].max()) & (xe["bn"] == "update")]
    rf = rf.merge(base, on=ids).assign(recovery=lambda d: d["mem_gap"] - d["before"])
    print("retain fine-tuning (5 epochs): mem_gap after minus before (retrain = the never-trained reference):")
    display(rf.groupby("method")["recovery"].agg(["mean", "count"]).round(4))
hb = U.load_df("headbias", CFG)
if len(hb):
    print("head-bias check (class level), means:")
    display(hb.groupby(["kind", "method"])[["bias_z", "bias_z_orig", "bias_restore_forget_acc", "row_restore_forget_acc", "shift_recall", "shift_fpr"]].mean().round(3))

## Next step

Save a version. Then open **notebook 08** (CPU) and attach the outputs of notebooks 01-07.

## Name of this run's output

Title this notebook **`ext-07-example-level`** and create a dataset with the same name from its output (viewer page
→ **Output** → **New Dataset**, Private), or attach the notebook output directly.

| Output name | Contains | Attached by |
|---|---|---|
| `ext-07-example-level` | `headbias`, `xl_*` results, `xl_calibration.json`, 10 retrained + 50 unlearned example-level checkpoints (≈ 1.3 GB) | notebook 08 |

- **Re-runs and resumes:** add a **New Version** of the same dataset; finished jobs are skipped and carried over.
- **Smoke (dry) runs:** add `-smoke`.